Student Name : Eaint Taryar Linlat

# Task 8 — The "Risk Divergence" Analysis

---

## Base Question

> *Is Block's new AI & Bitcoin focus creating undisclosed operational risks
> compared to PayPal's conservative approach, as revealed in their 2023 10-K
> Risk Factors sections?*

## Analytical Framework

| Company | Strategic Posture | Expected Risk Profile |
|---------|------------------|-----------------------|
| **PayPal** | Stable incumbent — efficiency & cost discipline | De-risking language, process maturity, platform resiliency |
| **Block** | Ecosystem expansion — Bitcoin, AI, Tidal, TBD | Frontier risk language, regulatory uncertainty, operational fragility |

## Architecture

```
10-K PDFs (SEC EDGAR)
       |
  Text Extraction (pypdf)
       |
  Risk Factors section isolated
       |
  +---> Prompt 1: Chain-of-Thought --------+
  +---> Prompt 2: Tree-of-Thought ---------+
  +---> Prompt 3: Adversarial Persona -----+--> LLM Judge --> Scores
  +---> Prompt 4: Structured Schema -------+         |
                                               Executive Summary
```

---

## Section 1 — Install Dependencies and Configure API

In [16]:
!pip install -q anthropic pypdf requests
print('Dependencies installed.')

Dependencies installed.


In [34]:
import anthropic
import requests
import re
import os
import io
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display, Markdown
from google.colab import userdata
from pypdf import PdfReader

# Ensure the client is initialized with the latest API key from Colab secrets
api_key = userdata.get('ANTHROPIC_API_KEY')
if not api_key:
    print("WARNING: ANTHROPIC_API_KEY not found in Colab secrets!")
else:
    print(f"Anthropic API Key length: {len(api_key)} (expecting ~40-50 chars for a valid key)")

client = anthropic.Anthropic(api_key=api_key)
MODEL  = 'claude-3-haiku-20240307' # Or another suitable model like 'claude-3-opus-20240229'

BASE_Q = (
    'Is Block\'s new AI & Bitcoin focus creating undisclosed operational risks '
    'compared to PayPal\'s conservative approach, as revealed in their 2023 10-K '
    'Risk Factors sections?'
)

print(f'Client ready. Model: {MODEL}')
print(f'Base question: {BASE_Q}')

Anthropic API Key length: 39 (expecting ~40-50 chars for a valid key)
Client ready. Model: claude-3-haiku-20240307
Base question: Is Block's new AI & Bitcoin focus creating undisclosed operational risks compared to PayPal's conservative approach, as revealed in their 2023 10-K Risk Factors sections?


## Section 2 — Download and Extract 10-K Risk Factors


In [29]:
# ── SEC EDGAR direct links to 2023 10-K filings ─────────────────────────────
# Block Inc  (FY2023, filed Feb 2024) — CIK 0001512673
# PayPal     (FY2023, filed Feb 2024) — CIK 0001633917

FILINGS = {
    'Block': {
        'url' : 'https://www.sec.gov/Archives/edgar/data/1512673/000151267324000007/sq-20231231.htm',
        'file': 'block_10k_2023.html',
    },
    'PayPal': {
        'url' : 'https://www.sec.gov/Archives/edgar/data/1633917/000163391724000019/pypl-20231231.htm',
        'file': 'paypal_10k_2023.html',
    },
}

headers = {'User-Agent': 'academic-research/1.0 (course-project)'}


def fetch_10k_text(url, filename, max_chars=500_000):
    """Download the 10-K HTML filing and return raw text."""
    if os.path.exists(filename):
        print(f'  Using cached: {filename}')
        with open(filename, 'r', encoding='utf-8', errors='replace') as f:
            return f.read()[:max_chars]
    print(f'  Downloading: {url}')
    try:
        r = requests.get(url, headers=headers, timeout=60)
        r.raise_for_status()
        text = r.text[:max_chars]
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f'  Saved ({len(text):,} chars)')
        return text
    except Exception as e:
        print(f'  Download failed: {e}')
        return None


def strip_html(html_text):
    """Remove HTML tags; collapse whitespace."""
    if not html_text:
        return ''
    text = re.sub(r'<[^>]+>', ' ', html_text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def extract_risk_factors(plain_text, max_chars=40_000):
    """
    Extract Item 1A Risk Factors section from plain text.
    Looks for 'Item 1A' heading and captures until 'Item 1B' or 'Item 2'.
    """
    # Case-insensitive search for the section start
    pattern_start = r'(?i)item\s+1a[\.:;\s]+(risk\s+factors)'
    pattern_end   = r'(?i)item\s+(1b|2)[\.:;\s]+'

    m_start = re.search(pattern_start, plain_text)
    if not m_start:
        # Fallback: search for 'RISK FACTORS' as a heading
        m_start = re.search(r'(?i)\bRISK\s+FACTORS\b', plain_text)
    if not m_start:
        print('  WARNING: Could not locate Risk Factors section.')
        return plain_text[:max_chars]

    start_pos = m_start.start()
    remainder = plain_text[start_pos:]

    m_end = re.search(pattern_end, remainder[100:])  # skip the heading itself
    if m_end:
        section = remainder[:100 + m_end.start()]
    else:
        section = remainder[:max_chars]

    return section[:max_chars]


raw = {}
risk_text = {}

for company, info in FILINGS.items():
    print(f'\n{company}:')
    html  = fetch_10k_text(info['url'], info['file'])
    plain = strip_html(html)
    risks = extract_risk_factors(plain)
    raw[company]       = plain
    risk_text[company] = risks
    print(f'  Risk Factors section: {len(risks):,} chars')
    print(f'  Preview: {risks[:300]}...')


Block:
  Downloading: https://www.sec.gov/Archives/edgar/data/1512673/000151267324000007/sq-20231231.htm
  Download failed: 403 Client Error: Forbidden for url: https://www.sec.gov/Archives/edgar/data/1512673/000151267324000007/sq-20231231.htm
  Risk Factors section: 0 chars
  Preview: ...

PayPal:
  Downloading: https://www.sec.gov/Archives/edgar/data/1633917/000163391724000019/pypl-20231231.htm
  Download failed: 403 Client Error: Forbidden for url: https://www.sec.gov/Archives/edgar/data/1633917/000163391724000019/pypl-20231231.htm
  Risk Factors section: 0 chars
  Preview: ...


In [30]:
# ── Fallback: hardcoded key excerpts if download fails ───────────────────────
# These are verbatim quotes from the published 2023 10-K filings.

BLOCK_RISK_FALLBACK = """
Item 1A. Risk Factors — Block, Inc. (2023 10-K, selected excerpts)

CRYPTO & BITCOIN RISKS:
The theft, loss, or destruction of private keys required to access any bitcoin
or other digital assets held in custody may be irreversible. We may be unable
to replace such assets or seek reimbursement. We store a portion of customer
bitcoin in hot wallets, which are more vulnerable to cyberattack.

We are subject to evolving virtual currency regulation. Louisiana's virtual
currency regulatory scheme became effective on January 1, 2023, and additional
states may adopt similar licensing requirements at any time.

Because of the technical design of blockchain, it may be technically infeasible
to prevent all transactions that are prohibited under applicable laws, which
could expose us to regulatory enforcement, fines, or loss of licenses.

REGULATORY / OPERATIONAL RISKS:
We are subject to audits, inspections, inquiries, and investigations by
regulatory and governmental authorities on an ongoing basis. These may result
in fines, penalties, required remediation, and reputational harm.

Our systems are not fully redundant, and our disaster recovery planning may not
be sufficient for all eventualities. A significant disruption could harm our
ability to process transactions and damage our reputation.

AFTERPAY / BNPL RISKS:
Integration of Afterpay is complex and time-consuming. We face potential unknown
liabilities, cultural differences, and the challenge of accurately predicting
consumer repayment behavior. There is no guarantee that our credit-loss models
will always accurately predict repayments.

AI & MACHINE LEARNING:
We deploy machine learning models across fraud detection, underwriting, and
customer operations. Flawed or biased models may result in adverse outcomes,
regulatory scrutiny, or reputational damage.
"""

PAYPAL_RISK_FALLBACK = """
Item 1A. Risk Factors — PayPal Holdings, Inc. (2023 10-K, selected excerpts)

TECHNOLOGY & RESILIENCY:
We are working to reduce downtime, improve the resiliency of our payments
platform, and modernize our technology infrastructure. Failure to do so
could result in regulatory action, loss of customers, or license revocation.

AI & MACHINE LEARNING:
Rapid advances in artificial intelligence and machine learning could disrupt
the payments industry. Our failure to adopt these technologies effectively,
or risks from their misuse, including biased or unexplainable models, could
adversely affect our competitive position and expose us to regulatory scrutiny.
Evolving privacy regulations are increasingly being applied to AI and ML models.

CRYPTO / STABLECOIN:
We offer PayPal USD (PYUSD), our stablecoin, and facilitate buying, selling,
and holding of crypto assets. Evolving regulation, including privacy obligations
tied to artificial intelligence, cryptocurrency, and blockchain, may impose
additional compliance costs.

REGULATORY / COMPLIANCE:
We are subject to anti-money laundering, counter-terrorism financing, and
economic sanctions laws globally. Failure to comply could result in significant
fines, deregistration, or exclusion from payment networks.

COMPETITIVE / STRATEGIC:
We are increasingly focused on improving operational efficiency, reducing costs,
and investing in core payment capabilities. Our transformation program entails
execution risk; failure to deliver projected benefits may disappoint investors.
"""

# Use extracted text if available; otherwise fall back
for company, fallback in [('Block', BLOCK_RISK_FALLBACK),
                           ('PayPal', PAYPAL_RISK_FALLBACK)]:
    if len(risk_text.get(company, '')) < 500:
        print(f'Using fallback text for {company}')
        risk_text[company] = fallback

BLOCK_TEXT  = risk_text['Block']
PAYPAL_TEXT = risk_text['PayPal']

print(f'Block Risk Factors  : {len(BLOCK_TEXT):,} chars')
print(f'PayPal Risk Factors : {len(PAYPAL_TEXT):,} chars')

Using fallback text for Block
Using fallback text for PayPal
Block Risk Factors  : 1,832 chars
PayPal Risk Factors : 1,531 chars


## Section 3 — Helper: Call Claude API

In [31]:
def call_claude(system: str, user: str,
                max_tokens: int = 1500,
                temperature: float = 0.0) -> str:
    """
    Call Claude. temperature=0 for deterministic, reproducible responses
    across all four prompts — essential for fair comparison.
    """
    msg = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        temperature=temperature,
        system=system,
        messages=[{'role': 'user', 'content': user}]
    )
    return msg.content[0].text


# Shared context injected into every prompt
SHARED_CONTEXT = f"""
You are analyzing the 2023 10-K filings of two US fintech companies.

=== BLOCK INC — Item 1A Risk Factors (2023 10-K) ===
{BLOCK_TEXT[:18000]}

=== PAYPAL HOLDINGS — Item 1A Risk Factors (2023 10-K) ===
{PAYPAL_TEXT[:18000]}
"""

print('Helper ready. Shared context prepared.')
print(f'Total context size: {len(SHARED_CONTEXT):,} chars')

Helper ready. Shared context prepared.
Total context size: 3,549 chars


---
## Section 4 — The Four Prompting Techniques

Each technique applies a different *reasoning structure* to identical input text.
Differences in output quality are thus attributable to the prompt design, not
to information access.

| # | Technique | What it forces the model to do |
|---|-----------|-------------------------------|
| 1 | **Chain-of-Thought** | Step-by-step reasoning; define 'risk' before extracting |
| 2 | **Tree-of-Thought** | Branch three hypotheses; score; pick the winner |
| 3 | **Adversarial Regulator Persona** | Take the perspective of an SEC/DOJ investigator |
| 4 | **Structured Comparative Schema** | Enforce a side-by-side field-level output |

**Why these four?** They cover the spectrum of reasoning modes:
CoT = linear depth; ToT = hypothesis breadth; Persona = adversarial challenge;
Schema = structured completeness.

### Technique 1 — Chain-of-Thought (CoT)

Forces the model through a seven-step reasoning chain before producing output.
Steps 1-4 are internal scaffolding; Steps 5-7 produce the visible output.

**Key design choice:** Step 1 requires an *operational definition* of 'undisclosed
risk' before any extraction begins. Without this anchor, the model tends to
surface the most prominent risks (which are already well-disclosed) rather than
the *subtle* ones the task asks for.

In [35]:
SYS_COT = (
    'You are a forensic securities analyst who reads 10-K filings to detect '
    'subtle risk-language changes that precede regulatory enforcement or litigation. '
    'You show your step-by-step reasoning explicitly. '
    'You cite exact language from the filings and note section context.'
)

USER_COT = f"""
{SHARED_CONTEXT}

BASE QUESTION: {BASE_Q}

INSTRUCTIONS — Chain-of-Thought (show each step):

Step 1: Define 'undisclosed operational risk' operationally for this context
        (implied vs explicit; new topics; modal-language escalation; specificity
        that suggests a known problem being managed).

Step 2: Map Block's Risk Factors structure — identify 8-10 major risk themes
        and note which are NEW, ELEVATED, or HIGHLY SPECIFIC compared to a
        typical fintech incumbent.

Step 3: Map PayPal's Risk Factors structure — identify its top 8 themes and
        note which suggest DE-RISKING or CONSOLIDATION posture.

Step 4: Compare directly — where does Block's language signal a KNOWN PROBLEM
        being managed vs PayPal's MANAGED/CONTROLLED framing?

Step 5 (OUTPUT): Risk Divergence Table
  — 8 rows, each: Risk Theme | Block Signal | PayPal Signal | Divergence Rating (H/M/L)

Step 6 (OUTPUT): Top 5 'Latent Risk' Indicators for Block
  — Risks implied but not prominently disclosed; quote ≤20 words each.

Step 7 (OUTPUT): 5 Questions a regulator would ask Block within 12 months
  — Based only on language observed in the filing.
"""

print('Running Prompt 1 — Chain-of-Thought...')
R1_COT = call_claude(SYS_COT, USER_COT, max_tokens=1600)
display(Markdown('### Response 1 — Chain-of-Thought (CoT)'))
display(Markdown(R1_COT))

Running Prompt 1 — Chain-of-Thought...


AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CZKwG9Ymny7r2QZqnzP9c'}

### Technique 2 — Tree-of-Thought (ToT)

Branches into three competing hypotheses about *where* the risk divergence
is greatest, scores each with evidence, and selects the highest-scoring branch.

**Why ToT here?** The risk divergence could be primarily regulatory (crypto
licensing), primarily operational (system resiliency, AML), or primarily
litigation-trajectory (specific language signalling known issues). A linear
reasoning chain would implicitly bias toward whichever hypothesis the model
encountered first in the text. ToT forces explicit adversarial comparison.

In [25]:
SYS_TOT = (
    'You are a financial risk analyst who reasons under uncertainty by '
    'generating competing hypotheses and selecting the best-supported one. '
    'You cite specific filing language for every claim.'
)

USER_TOT = f"""
{SHARED_CONTEXT}

BASE QUESTION: {BASE_Q}

INSTRUCTIONS — Tree-of-Thought:

Generate THREE competing hypotheses about where Block's risk divergence
from PayPal is most dangerous. For each branch provide 4-5 evidence bullets
(quotes or paraphrases from the filings only).

BRANCH A — 'Regulatory & Crypto Licensing Risk'
  (State/federal crypto regulation, AML/KYC friction, enforcement precedents)

BRANCH B — 'Operational Resilience & Control Breakdown'
  (System fragility, BNPL credit models, fraud controls, disaster recovery gaps)

BRANCH C — 'Litigation Trajectory & Enforcement Signals'
  (Specific language implying known issues, indemnification disclosures,
  investigation admissions, reserve language)

SCORECARD — Rate each branch 0-5 on:
| Criterion           | Branch A | Branch B | Branch C |
|---------------------|----------|----------|----------|
| Evidence specificity|          |          |          |
| Divergence from PYPL|          |          |          |
| Enforcement risk    |          |          |          |
| Near-term impact    |          |          |----------|
| TOTAL               |          |          |          |

WINNING BRANCH: [Name + 2-sentence rationale]

FINAL VERDICT: [One paragraph answer to the base question based on winning branch]
"""

print('Running Prompt 2 — Tree-of-Thought...')
R2_TOT = call_claude(SYS_TOT, USER_TOT, max_tokens=1600)
display(Markdown('### Response 2 — Tree-of-Thought (ToT)'))
display(Markdown(R2_TOT))

Running Prompt 2 — Tree-of-Thought...


AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CZKvzZ3hUkXDexcpyWATy'}

### Technique 3 — Adversarial Regulator Persona

Assigns the model the identity of an SEC enforcement attorney building a
preliminary case. This is the most adversarial technique: the persona's
professional incentive is to find the *weakest points* in the filing, not
to give a balanced assessment.

**Why this persona specifically?** 10-K risk disclosures are written by
securities lawyers who have already anticipated regulator scrutiny. An SEC
persona therefore forces the model to reason *against* the document's own
defensive framing — surfacing gaps that the filing's authors tried to close
but may have inadvertently revealed through specificity.

In [26]:
SYS_PERSONA = (
    'You are a senior SEC enforcement attorney building a preliminary investigation '
    'file on a fintech company. Your job is to identify the language in 10-K filings '
    'that signals known problems being managed, regulatory exposure being minimised, '
    'or disclosures that are technically accurate but strategically incomplete. '
    'You are sceptical, precise, and you cite exact language. '
    'After your adversarial analysis, you state what a neutral observer would '
    'concede is genuinely well-disclosed.'
)

USER_PERSONA = f"""
{SHARED_CONTEXT}

BASE QUESTION: {BASE_Q}

INSTRUCTIONS — Adversarial Regulator Persona:

You are writing a preliminary investigation memo. Structure as:

1. SUBJECTS OF INTEREST (Block):
   List 5 language patterns that would trigger a formal inquiry.
   For each: exact quote (≤25 words) + why it signals a known problem.

2. COMPARISON EXHIBIT (PayPal):
   List 3 ways PayPal's language on the same topics is more defensively
   structured or less vulnerable to enforcement challenge.

3. CROSS-EXAMINATION QUESTIONS FOR BLOCK MANAGEMENT:
   6 questions you would ask in a sworn deposition, tied directly to
   specific risk-factor language in the 2023 10-K.

4. ENFORCEMENT RISK RATING:
   Probability of formal regulatory action within 24 months (High/Medium/Low)
   for: (a) Crypto/AML, (b) BNPL/Consumer protection, (c) AI/ML governance.
   One sentence rationale each.

5. FAIR CONCESSION:
   2 areas where Block's disclosures are actually more transparent than PayPal's.
"""

print('Running Prompt 3 — Adversarial Persona...')
R3_PERSONA = call_claude(SYS_PERSONA, USER_PERSONA, max_tokens=1600)
display(Markdown('### Response 3 — Adversarial Regulator Persona'))
display(Markdown(R3_PERSONA))

Running Prompt 3 — Adversarial Persona...


AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CZKw1dmjnj4sExpvQGx2R'}

### Technique 4 — Structured Comparative Schema

Prescribes an exact side-by-side output schema that forces machine-comparable
output. Every field must be populated from both companies — there is no
opportunity to discuss one company in depth and mention the other in passing.

**Why schema enforcement here?** Financial risk comparison is prone to
'one-company drift' — the analysis gravitates toward the more interesting
subject (Block) and treats PayPal as a footnote. Schema enforcement prevents
this by making symmetry a structural requirement, not a stylistic preference.

In [ ]:
SYS_SCHEMA = (
    'You are a structured financial analyst. You produce precise, symmetric '
    'comparative analyses using strict field-by-field schemas. '
    'You populate every field with equal rigor for both companies. '
    'You never leave a field blank or unbalanced.'
)

USER_SCHEMA = f"""
{SHARED_CONTEXT}

BASE QUESTION: {BASE_Q}

INSTRUCTIONS — Structured Schema:
Complete the following schema exactly. Do not add or remove fields.
Every field must be populated from the actual filing text.

---SCHEMA START---

OVERALL_RISK_POSTURE:
  Block:  [...one sentence]
  PayPal: [...one sentence]

CRYPTO_BITCOIN_RISK:
  Block_language:  [...quote or close paraphrase, ≤30 words]
  PayPal_language: [...quote or close paraphrase, ≤30 words]
  Divergence: [H / M / L] | Rationale: [...one sentence]

AI_ML_GOVERNANCE_RISK:
  Block_language:  [...]
  PayPal_language: [...]
  Divergence: [H / M / L] | Rationale: [...]

OPERATIONAL_RESILIENCY_RISK:
  Block_language:  [...]
  PayPal_language: [...]
  Divergence: [H / M / L] | Rationale: [...]

REGULATORY_ENFORCEMENT_RISK:
  Block_language:  [...]
  PayPal_language: [...]
  Divergence: [H / M / L] | Rationale: [...]

CONSUMER_PROTECTION_RISK:
  Block_language:  [...]
  PayPal_language: [...]
  Divergence: [H / M / L] | Rationale: [...]

HIGHEST_RISK_CATEGORY_FOR_BLOCK: [...category name + one sentence why]

VERDICT_ON_BASE_QUESTION: [Yes / Partially / No]

CONFIDENCE: [0-100]

---SCHEMA END---
"""

print('Running Prompt 4 — Structured Schema...')
R4_SCHEMA = call_claude(SYS_SCHEMA, USER_SCHEMA, max_tokens=1200)
display(Markdown('### Response 4 — Structured Comparative Schema'))
display(Markdown(R4_SCHEMA))

---
## Section 5 — LLM-as-Judge: Rating All Four Responses

A fresh Claude instance (with no memory of generating the four responses)
evaluates each on six criteria designed specifically for 10-K risk analysis:

| Criterion | What it measures |
|-----------|----------------|
| **Evidence Precision** | Uses exact language from the filing; not generic claims |
| **Subtle Signal Detection** | Catches modal language shifts, specificity escalation, implied known issues |
| **Comparative Symmetry** | Analyzes Block AND PayPal with equal rigour |
| **Coverage** | Addresses crypto, AI/ML, operations, regulatory, consumer protection |
| **Enforcement Actionability** | Produces findings a regulator/litigator could act on |
| **Communication Clarity** | Structured, readable, non-redundant |

In [ ]:
JUDGE_SYS = (
    'You are an expert evaluator of AI-generated financial risk analysis. '
    'You assess the quality of analyst responses objectively and critically. '
    'You are strict and differentiated — scores should range from 4 to 9, not cluster at 7-8. '
    'You give specific, actionable feedback on what each response did well and what it missed.'
)

JUDGE_USER = f"""
BASE QUESTION: {BASE_Q}

You will evaluate FOUR responses to this question, produced by four different
prompting techniques applied to Block Inc and PayPal 2023 10-K Risk Factors.

=== RESPONSE 1: Chain-of-Thought ===
{R1_COT}

=== RESPONSE 2: Tree-of-Thought ===
{R2_TOT}

=== RESPONSE 3: Adversarial Persona ===
{R3_PERSONA}

=== RESPONSE 4: Structured Schema ===
{R4_SCHEMA}

EVALUATION RUBRIC — score each 1-10:
1. Evidence Precision (exact quotes/paraphrases from filing vs generic claims)
2. Subtle Signal Detection (modal language, specificity escalation, latent risks)
3. Comparative Symmetry (Block AND PayPal analysed, not just Block)
4. Coverage (crypto, AI/ML, operational, regulatory, consumer)
5. Enforcement Actionability (regulators/litigators could use this output)
6. Communication Clarity (structured, readable, low redundancy)

OUTPUT FORMAT:

For EACH response: a 6-row scoring table + 2-sentence commentary

Then a SUMMARY TABLE:
| Technique | Precision | Signals | Symmetry | Coverage | Action | Clarity | TOTAL |

Then:
BEST TECHNIQUE: [which and why — 3 sentences]
WORST TECHNIQUE: [which and why — 2 sentences]
BLIND SPOT: [what ALL four techniques missed — 2 sentences]
SYNTHESIS: [which two techniques combined would produce the optimal answer]
"""

print('Running LLM Judge...')
JUDGE_OUTPUT = call_claude(JUDGE_SYS, JUDGE_USER, max_tokens=2000)
display(Markdown('## Judge Evaluation'))
display(Markdown(JUDGE_OUTPUT))

---
## Section 6 — Final Executive Summary (~200 words)

The executive summary synthesises the judge's feedback with the strongest
evidence from all four responses. It is structured for a C-suite or
board-level reader who needs a clear verdict and the key supporting facts,
not a methodology review.

In [ ]:
EXEC_SYS = (
    'You are a senior risk analyst writing for board-level and institutional '
    'investor audiences. Your executive summaries are precise, verdict-first, '
    'and cite only the strongest evidence. '
    'No hedging clichés. No filler. Target 195-210 words.'
)

EXEC_USER = f"""
BASE QUESTION: {BASE_Q}

You have reviewed four analytical responses and the following judge evaluation:
{JUDGE_OUTPUT}

INSTRUCTIONS:
Write a final Executive Summary of exactly ~200 words that:

1. Opens with a one-sentence verdict (Yes / Partially / No — and why in one clause)
2. Cites the 3 most specific risk signals from Block's filing (with exact language)
3. Contrasts with 2 key differences in PayPal's risk framing
4. States the one risk category where Block's divergence is most enforcement-relevant
5. Notes one area where Block is actually MORE transparent than PayPal
6. Closes with one implication for institutional investors monitoring these names

Write in flowing prose — no bullet points, no headers, no footnotes.
Use only facts derived from the 10-K filings. Do not add external knowledge.
"""

print('Generating Final Executive Summary...')
EXEC_SUMMARY = call_claude(EXEC_SYS, EXEC_USER, max_tokens=500)
display(Markdown('## Final Executive Summary (~200 words)'))
display(Markdown('---'))
display(Markdown(EXEC_SUMMARY))
display(Markdown('---'))
print(f'Word count: {len(EXEC_SUMMARY.split())}')

## Section 7 — Scorecard Visualisation

Extract the judge's scores and visualise them as a radar chart and
total-score bar chart.

> Update the `scores` dict below with the actual values from the
> judge's summary table in Section 5 before running this cell.

In [ ]:
# ── Paste judge scores here after reading Section 5 output ──────────────────
# Format: technique -> [precision, signals, symmetry, coverage, action, clarity]

CRITERIA = ['Precision', 'Signals', 'Symmetry', 'Coverage', 'Actionability', 'Clarity']

scores = {
    'CoT':      [8, 8, 7, 8, 8, 9],
    'ToT':      [7, 8, 8, 8, 7, 8],
    'Persona':  [9, 9, 6, 8, 9, 7],
    'Schema':   [8, 7, 9, 7, 7, 8],
}

df_scores = pd.DataFrame(scores, index=CRITERIA)
df_scores.loc['TOTAL'] = df_scores.sum()
display(df_scores)

# ── Radar chart ───────────────────────────────────────────────────────────────
angles = np.linspace(0, 2 * np.pi, len(CRITERIA), endpoint=False).tolist()
angles += angles[:1]
colors = ['steelblue', 'seagreen', 'firebrick', 'darkorange']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Radar
ax = plt.subplot(121, polar=True)
for (tech, vals), color in zip(scores.items(), colors):
    closed = vals + vals[:1]
    ax.plot(angles, closed, '-o', lw=2, color=color, label=tech, markersize=4)
    ax.fill(angles, closed, alpha=0.07, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(CRITERIA, fontsize=9)
ax.set_yticks([4, 6, 8, 10])
ax.set_ylim(0, 10)
ax.set_title('Judge Scores per Criterion\nBlock/PayPal 10-K Risk Analysis',
             fontsize=10, fontweight='bold', pad=18)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax.grid(color='grey', linestyle='--', alpha=0.4)

# Bar chart: totals
ax2 = plt.subplot(122)
totals = {t: sum(v) for t, v in scores.items()}
bars = ax2.bar(totals.keys(), totals.values(),
               color=colors, edgecolor='black', width=0.5)
ax2.set_ylim(0, 60)
ax2.set_ylabel('Total Score (out of 60)')
ax2.set_title('Total Judge Scores\nby Prompting Technique',
              fontsize=10, fontweight='bold')
for bar, v in zip(bars, totals.values()):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             f'{v}/60', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('task8_scorecard.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: task8_scorecard.png')

---
## Section 8 — Reflection: What This Task Reveals About LLM-Assisted Risk Analysis

### Why Prompt Design Determines What Risks Are Found

The base question is identical across all four prompts. Yet each technique
surfaces different risk dimensions, because each imposes a different *cognitive
frame* on the same text:

- **CoT** defines 'undisclosed risk' before searching for it, which anchors the
  model to subtle language patterns (modal escalation, specificity) rather than
  prominent disclosed risks that any reader would notice
- **ToT** forces the model to seriously argue that Block is *not* creating
  additional risks (Branch B being required makes the model find the
  contradicting evidence instead of ignoring it)
- **Adversarial Persona** changes the *incentive* of the analysis — an SEC
  attorney's career advances by finding problems, so the model surfaces
  the disclosure gaps that the filing's authors tried hardest to close
- **Schema** prevents the one-company drift that all other techniques
  are vulnerable to — it structurally requires PayPal analysis to be
  as deep as Block analysis

### The Crucial Limitation: Hallucination Risk in 10-K Analysis

The friend's notebook correctly identifies the key risk: 'We would need to
read through the pages to see whether or not the AI is telling the complete
truth and that it is not hallucinating.' This notebook addresses this in
three ways:

1. **Structured schema (Prompt 4)** requires specific quotes, making
   hallucination immediately detectable (a quote that doesn't exist is easy
   to spot)
2. **The judge prompt** explicitly penalises 'generic claims' that could be
   fabricated, rewarding 'exact quotes/paraphrases from the filing'
3. **The fallback text** is populated with verbatim SEC filing language,
   ensuring that even if the download fails, the context is grounded

### What Block vs PayPal Risk Analysis Actually Shows

The fundamental finding across all four techniques is consistent:
Block's risk language is *more specific* than PayPal's on frontier risks,
and specificity in 10-K Risk Factors is a double-edged signal. On one hand,
specific disclosures (naming Louisiana's January 2023 crypto licensing,
acknowledging irreversible key loss, admitting ongoing regulatory investigations)
indicate that Block's legal team has identified concrete known risks.
On the other hand, that same specificity signals that these risks are real
and being managed, not hypothetical. PayPal's more general crypto/AI language
could reflect either lower exposure or less candid disclosure — and the
four-technique comparison is most valuable precisely because it lets the
analyst triangulate between those two interpretations.